# Análise de Cohort para Retenção de Clientes

Este notebook realiza a análise de cohort para entender a retenção de clientes ao longo do tempo, utilizando o dataset Online Retail II.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import os
import config

## 1. Carregar o dataset

In [ ]:
# Carregar o dataset
try:
    file_path = os.path.join(config.PATH, 'online_retail_II.csv')
    df = pd.read_csv(file_path)
    print("Dataset carregado com sucesso!")
    display(df.head())
except FileNotFoundError:
    print(f"Arquivo não encontrado em: {file_path}")

## 2. Limpeza dos Dados

In [ ]:
# Verificar nulos
print("Nulos antes da limpeza:")
print(df.isnull().sum())

# Remover nulos (especialmente Customer ID)
df = df.dropna(subset=['Customer ID'])

# Remover cancelamentos (Invoice contendo 'C')
df = df[~df['Invoice'].astype(str).str.contains('C')]

# Remover valores negativos ou zerados em Quantity e Price
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# Converter InvoiceDate para datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Converter Customer ID para inteiro
df['Customer ID'] = df['Customer ID'].astype(int)

print("\nDimensões após limpeza:", df.shape)

## FASE 1: Definição de Cohort
Nesta fase, vamos:
1. Identificar o mês da transação (`TransactionMonth`).
2. Identificar o mês da primeira compra de cada cliente (`CohortMonth`).
3. Calcular a idade do cohort (`CohortIndex`), que representa o número de meses desde a primeira compra.

In [ ]:
# Função para obter o primeiro dia do mês
def get_month(x):
    return dt.datetime(x.year, x.month, 1)

# Criar coluna TransactionMonth
df['TransactionMonth'] = df['InvoiceDate'].apply(get_month)

# Identificar CohortMonth (mês da primeira compra para cada cliente)
grouping = df.groupby('Customer ID')['TransactionMonth']
df['CohortMonth'] = grouping.transform('min')

df.head()

In [ ]:
# Função para extrair ano, mês e dia
def get_date_int(df, column):
    year = df[column].dt.year
    month = df[column].dt.month
    day = df[column].dt.day
    return year, month, day

# Extrair ano e mês
invoice_year, invoice_month, _ = get_date_int(df, 'TransactionMonth')
cohort_year, cohort_month, _ = get_date_int(df, 'CohortMonth')

# Calcular a diferença em anos e meses
years_diff = invoice_year - cohort_year
months_diff = invoice_month - cohort_month

# Calcular CohortIndex (idade do cohort em meses, começando de 1)
df['CohortIndex'] = years_diff * 12 + months_diff + 1

df.head()